# Model performance and interpretability over time

This notebook **only reads** synced artifacts under `results/` (history, metrics, permutation importance, PDP images). It does **not** retrain models or call GCP.

**Colab:** run all cells; the default clone URL is this course repository. Override with environment variable `NOTEBOOK_REPO_URL` if you use a fork.

In [ ]:
# Clone repo on Colab; locally, use the checkout that contains results/
import os
import sys

DEFAULT_REPO = "https://github.com/OPIM5512-mjb24001/myscrapers-mjb24001-v3.git"
REPO_URL = os.environ.get("NOTEBOOK_REPO_URL", DEFAULT_REPO)
BRANCH = os.environ.get("NOTEBOOK_REPO_BRANCH", "main")

if "google.colab" in sys.modules:
    import subprocess
    subprocess.run(["rm", "-rf", "repo"], check=False)
    subprocess.run(
        ["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, "repo"],
        check=True,
    )
    os.chdir("repo")
    print(f"Cloned: {REPO_URL} (branch {BRANCH})")
else:
    if os.path.isdir("results"):
        print("Using current working directory as repo root.")
    elif os.path.isdir(os.path.join("..", "results")):
        os.chdir("..")
        print("Using parent directory as repo root.")
    else:
        print("Tip: open the notebook from the repository root so ./results exists, or run in Colab.")

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except ImportError:
    display = print

ROOT = Path(".").resolve()
RESULTS = ROOT / "results"
hist = pd.DataFrame()

if not RESULTS.is_dir():
    print(
        f"No '{RESULTS.name}/' folder here. Run from repo root after sync, or use Colab to clone the project repo."
    )
else:
    hist_path = RESULTS / "history" / "metrics_history.csv"
    if hist_path.exists():
        hist = pd.read_csv(hist_path)
        if "timestamp_utc" in hist.columns:
            hist["timestamp_utc"] = pd.to_datetime(
                hist["timestamp_utc"], utc=True, errors="coerce"
            )
        display(hist.tail(10))
    else:
        print(
            "No results/history/metrics_history.csv yet. Run training (not dry_run) and the Sync model artifacts workflow."
        )

In [ ]:
# Trend: MAE, MAPE, RMSE, bias vs training run (one row per Cloud Function train job)
if hist.empty:
    print("Skip metric plots: no metrics history.")
else:
    if "run_id" in hist.columns:
        x = hist["run_id"].astype(str).tolist()
        xlabel = "run_id (UTC hour bucket)"
    elif "eval_date_local" in hist.columns:
        x = hist["eval_date_local"].astype(str).tolist()
        xlabel = "eval_date_local (holdout day, America/New_York)"
    else:
        x = list(range(len(hist)))
        xlabel = "row order in metrics_history.csv"

    fig, axes = plt.subplots(2, 2, figsize=(11, 8))
    metrics = [
        ("mae", "MAE"),
        ("mape", "MAPE (%)"),
        ("rmse", "RMSE"),
        ("bias", "Bias (mean pred − actual)"),
    ]
    for ax, (col, title) in zip(axes.ravel(), metrics):
        if col in hist.columns:
            ax.plot(x, hist[col].astype(float), marker="o", markersize=3)
            ax.set_title(title)
            ax.set_xlabel(xlabel)
            ax.tick_params(axis="x", rotation=45)
            ax.grid(True, alpha=0.3)
        else:
            ax.set_visible(False)
    plt.tight_layout()
    plt.show()

In [ ]:
# Permutation importance: overlay recent runs (timestamped CSVs under results/permutation_importance/)
import glob

if not RESULTS.is_dir():
    print("Skip importance plot: results/ not found.")
else:
    imp_files = sorted(
        glob.glob(str(RESULTS / "permutation_importance" / "*-permutation_importance.csv"))
    )
    if not imp_files:
        print("No *-permutation_importance.csv files yet. Run sync after a training job.")
    else:
        last_three = imp_files[-3:]
        fig, ax = plt.subplots(figsize=(10, 6))
        for fp in last_three:
            dfi = pd.read_csv(fp)
            dfi = dfi.sort_values("importance_mean", ascending=True).tail(12)
            label = Path(fp).stem.replace("-permutation_importance", "")
            ax.barh(dfi["feature"], dfi["importance_mean"], alpha=0.5, label=label)
        ax.set_title("Top features (last few training runs)")
        ax.legend()
        plt.tight_layout()
        plt.show()

In [ ]:
# Partial dependence plots (results/pdp/*.png)
import glob

from IPython.display import Image, display as ipy_display

if not RESULTS.is_dir():
    print("Skip PDPs: results/ not found.")
else:
    pdp_files = sorted(glob.glob(str(RESULTS / "pdp" / "*.png")))
    if not pdp_files:
        print("No PNG files in results/pdp/ yet. Run training + sync when PDPs are generated.")
    else:
        for fp in pdp_files[-6:]:
            ipy_display(Image(filename=fp))

## Short interpretation (edit after you inspect plots)

- **Metrics drift**: If MAE/RMSE rise on the latest `eval_date_local`, the latest scrape day may be harder to predict (mix shift, outliers) or the model needs more signal.
- **MAPE** is sensitive to cheap listings; compare with MAE.
- **Bias** consistently positive means systematic over-pricing predictions; negative means under-pricing.
- **Permutation importance** shifts show which features the forest relies on as `listings_master_llm.csv` grows.
- **PDPs** summarize marginal effect of top features on the tuned RandomForest; steep slopes imply stronger sensitivity.

_Fill 2–3 sentences for your midterm describing how behavior changed over the runs you plotted._